# Weapon Detection — V3 Training

Cleaned final version of the original V3 training notebook.

Repeated setup, extraction, resume, continuation, and debugging cells
have been consolidated into one documented workflow.

### Final V3 configuration

- **Base model:** V2.1 `best.pt` (YOLO26m)
- **Training dataset:** V2.2 master dataset
- **Input size:** 1280 px
- **Classes:** Knife, Machette, Handgun, Longgun, Other
- **Seed:** 42
- **Final recorded training state:** E14
- **E15 partial run:** excluded from the final model

This notebook is intended for project documentation and reproducibility.


In [ ]:
# ============================================================
# V3 TRAINING — SETUP AND CONFIGURATION
# ============================================================

from pathlib import Path
import zipfile

from google.colab import drive
from ultralytics import YOLO

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")

V22_ZIP = DRIVE / "V2_Training_Dataset_V2.2.zip"

BASE_DIR = Path("/content/V3_V2_2")
DATASET_DIR = BASE_DIR / "V2_Training_Dataset_V2.2"

V21_BEST = (
    DRIVE
    / "Weapon_Detection_Training"
    / "V2.1"
    / "v2_1_yolo26m_1280_e8"
    / "weights"
    / "best.pt"
)

V3_PROJECT = DRIVE / "Weapon_Detection_Training" / "V3"
V3_NAME = "v3_yolo26m_1280"
V3_RUN = V3_PROJECT / V3_NAME

IMAGE_SIZE = 1280
EPOCHS = 14
BATCH = -1
SEED = 42

print("V2.2 dataset:", V22_ZIP)
print("V2.1 checkpoint:", V21_BEST)
print("V3 output:", V3_RUN)


In [ ]:
# ============================================================
# EXTRACT V2.2 MASTER DATASET
# ============================================================

BASE_DIR.mkdir(parents=True, exist_ok=True)

if DATASET_DIR.exists():
    print("✓ V2.2 dataset already available")
else:
    print("Extracting V2.2 master dataset...")

    with zipfile.ZipFile(V22_ZIP, "r") as zip_ref:
        zip_ref.extractall(BASE_DIR)

    print("✓ V2.2 dataset extracted")


In [ ]:
# ============================================================
# CREATE DATASET YAML
# ============================================================

yaml_path = DATASET_DIR / "dataset.yaml"

yaml_path.write_text(
    f"""path: {DATASET_DIR.as_posix()}

train: train/images
val: val/images
test: test/images

names:
  0: Knife
  1: Machette
  2: Handgun
  3: Longgun
  4: Other
"""
)

print("✓ dataset.yaml created")
print(yaml_path)


In [ ]:
# ============================================================
# VERIFY STARTING CHECKPOINT
# ============================================================

if not V21_BEST.exists():
    raise FileNotFoundError(
        f"V2.1 checkpoint was not found:\n{V21_BEST}\n"
        "Update V21_BEST if the Drive location has changed."
    )

print("✓ V2.1 best checkpoint found")


In [ ]:
# ============================================================
# TRAIN V3
# ============================================================
# V2.1 best.pt → V2.2 dataset → V3
# YOLO26m | 1280px | final recorded horizon E14
#
# The later partial E15 run is intentionally not used.
# ============================================================

model = YOLO(str(V21_BEST))

model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH,
    seed=SEED,
    pretrained=True,
    project=str(V3_PROJECT),
    name=V3_NAME,
    exist_ok=True,
    save=True,
    save_period=1,
)


In [ ]:
# ============================================================
# FINAL ARTIFACT LOCATIONS
# ============================================================

BEST_MODEL = V3_RUN / "weights" / "best.pt"
LAST_MODEL = V3_RUN / "weights" / "last.pt"
RESULTS = V3_RUN / "results.csv"

print("=" * 60)
print("V3 TRAINING COMPLETE")
print("=" * 60)
print("Best model:", BEST_MODEL)
print("Last model:", LAST_MODEL)
print("Results:", RESULTS)
